# Part 2 — Group Presentation & Notebook

**Group members:** 

- Dargys Perez : `Data Engineer`
  
- Cecilia Back : `Data Analyst`

- Marta Bielow : `Accounting Manager`
  
- Jessika Rosen : `Head of Finance` – accountable for regulatory compliance and overall financial processes, not directly involved in the operational analysis meeting

## Data Quality Issues
Compliance with the 10-day rule (delivery to invoice) was the focus of the work meeting.

**The following issues were raised:**
- Why does the delivery_confirmed_date in Deliveries.csv (originating from the logistics system/TMS) contain multiple date formats?
- What is the current situation regarding the time gap between delivery_confirmed_date and invoice_date? 
- Based on available historical data, how well are we complying with the 10-day rule today?
 

In [ ]:
# using pandas for data analysis
import pandas as pd

In [ ]:
# load the relevant data for the 10 rules analysis
invoice_line = pd.read_csv("invoice_line.csv")
deliveries = pd.read_csv("deliveries.csv")


FileNotFoundError: [Errno 2] No such file or directory: 'invoice_line.csv'

In [1]:
# Just for checking how the "invoice_date" looks like in invoice_date
invoice_line['invoice_date']
# data type in the table "invoice_line"
invoice_line.dtypes


NameError: name 'invoice_line' is not defined

In [ ]:
# Just for checking how the "invoice_date" looks like in delivery_confirm_date
deliveries['delivery_confirm_date']

0              2027-02-10
1              2027-01-13
2              2027-03-03
3     2027-03-19T11:33:00
4              19/03/2027
5              2027-02-19
6              2027-02-19
7              2027-03-05
8              2027-03-22
9              2027-02-08
10             2027-01-19
11             2027-01-17
12             2027-02-04
13             2027-02-02
14             2027-03-03
15             2027-01-19
16             2027-02-28
17             2027-02-03
18             2027-01-29
19    2027-03-02T11:01:00
20            24 Mar 2027
21             2027-02-11
22             2027-02-07
23             2027-02-02
24             2027-02-15
25             13-02-2027
26             2027-01-25
27             2027-02-07
28             2027-01-28
29             2027-03-01
30    2027-02-13T11:36:00
31             2027-02-25
32             2027-01-18
33             2027-03-08
34             2027-01-31
35    2027-02-18T12:09:00
36             2027-01-26
37             2027-03-19
38          

In [ ]:
# data type in the table "deliveries"
deliveries.dtypes

delivery_confirm_id      str
order_id                 str
delivery_confirm_date    str
delivery_country         str
dtype: object

In [ ]:
invoice_line.groupby("order_id")["invoice_date"].nunique().value_counts()

In [ ]:
invoice_line.groupby("order_id")["invoice_date"].value_counts()

In [25]:
#Normalize invoice_line data to one row per order_id
invoice_order = (
    invoice_line
    .drop_duplicates(subset=["order_id"])
    [["order_id", "invoice_date"]]
)

In [ ]:
# Ensure that the data is cleaned from duplicates.
invoice_order.value_counts()

In [27]:
#Join datasets
df = invoice_order.merge(
    deliveries,
    on="order_id",
    how="left"
)

In [28]:
#Convert to datetime
df["invoice_date"] = pd.to_datetime(df["invoice_date"])


In [29]:
# the delivery_confirm_date has data quality issues, the date is given in differents formats and I need first to normalize.
df["delivery_confirm_date"] = pd.to_datetime(
    df["delivery_confirm_date"],
    format="mixed",
    dayfirst=True,
    errors="coerce"
)

In [58]:
#Calculate the date difference
df["days_diff"] = (df["invoice_date"] - df["delivery_confirm_date"]).dt.days

#Categorize the result
df["status"] = "Compliant"
df.loc[df["days_diff"] > 10, "status"] = "Late"
df.loc[df["days_diff"].isna(), "status"] = "Missing"

df["status"].value_counts()

summary = pd.DataFrame({
    "count": df["status"].value_counts(),
    "percent": (df["status"].value_counts(normalize=True) * 100).round(2)
})
 
print(summary)
 

           count  percent
status                   
Compliant     34     68.0
Late          16     32.0


In [42]:
# How the "days_diff" are distributed in the dataset.
#df["days_diff"].value_counts().sort_index(ascending=False)
df["days_diff"].value_counts() \
    .sort_index(ascending=False) \
    .reset_index(name="occurrences") \
    .rename(columns={"index": "days_diff"})

,days_diff,occurrences
0,76,1
1,40,1
2,27,2
3,25,1
4,23,1
5,22,2
6,20,2
7,19,1
8,16,1
9,15,2


In [61]:
#Calculate the date difference
df["days_diff"] = (df["invoice_date"] - df["delivery_confirm_date"]).dt.days

#Categorize the result
df["status"] = "Fulfill 10-day-rule"
df.loc[df["days_diff"] > 10, "status"] = "Late"
df.loc[df["days_diff"] < 0, "status"] = "Negative date-diff"
df.loc[df["days_diff"].isna(), "status"] = "Missing"

df["status"].value_counts()

summary = pd.DataFrame({
    "count": df["status"].value_counts(),
    "percent": (df["status"].value_counts(normalize=True) * 100).round(2)
})
 
print(summary)
 

                     count  percent
status                             
Fulfill 10-day-rule     22     44.0
Late                    16     32.0
Negative date-diff      12     24.0


In [62]:
# Sorting out the orders with negative day_diff.
df[df["status"] == "Negative date-diff"]

,order_id,invoice_date,delivery_confirm_id,delivery_confirm_date,delivery_country,days_diff,status
6,ORD-2026-483219,2027-02-06,CONF-560965,2027-08-01,Netherlands,-176,Negative date-diff
9,ORD-2027-296460,2027-02-17,CONF-747653,2027-10-02,France,-227,Negative date-diff
10,ORD-2027-340184,2027-02-21,CONF-831485,2027-03-02,Sweden,-9,Negative date-diff
12,ORD-2027-881543,2027-03-24,CONF-699281,2027-06-03,Germany,-71,Negative date-diff
15,ORD-2027-126344,2027-03-17,CONF-381172,2027-12-03,France,-261,Negative date-diff
16,ORD-2026-339139,2027-02-24,CONF-871248,2027-05-02,France,-67,Negative date-diff
26,ORD-2027-966069,2027-02-17,CONF-941444,2027-10-02,Germany,-227,Negative date-diff
30,ORD-2027-599775,2027-03-17,CONF-260684,2027-08-03,Belgium,-139,Negative date-diff
36,ORD-2027-116776,2027-03-10,CONF-198879,2027-05-03,Netherlands,-54,Negative date-diff
42,ORD-2027-346967,2027-02-17,CONF-526176,2027-08-02,Netherlands,-166,Negative date-diff


In [59]:
order_line = pd.read_csv("order_line.csv")

#Normalize order_line data to one row per order_id
order_order = (
    order_line
    .drop_duplicates(subset=["order_id"])
    [["order_id", "order_date"]]
    )

df1 = order_order.merge(
    df,
    on= "order_id",
    how="left"
)

In [56]:
#df1[["status","order_id", "order_date", "invoice_date", "delivery_confirm_date", "delivery_country", "days_diff"]]

df1.loc[df1["status"] == "Negative date-diff",
        ["status","order_id","order_date","invoice_date",
         "delivery_confirm_date","delivery_country","days_diff"]] \
   .sort_values("days_diff")

,status,order_id,order_date,invoice_date,delivery_confirm_date,delivery_country,days_diff
7,Negative date-diff,ORD-2027-126344,2027-02-28,2027-03-17,2027-12-03,France,-261.0
12,Negative date-diff,ORD-2027-911031,2027-01-19,2027-02-16,2027-11-02,Germany,-259.0
26,Negative date-diff,ORD-2027-296460,2027-01-31,2027-02-17,2027-10-02,France,-227.0
42,Negative date-diff,ORD-2027-966069,2027-01-09,2027-02-17,2027-10-02,Germany,-227.0
32,Negative date-diff,ORD-2026-483219,2026-12-23,2027-02-06,2027-08-01,Netherlands,-176.0
60,Negative date-diff,ORD-2027-346967,2027-01-05,2027-02-17,2027-08-02,Netherlands,-166.0
50,Negative date-diff,ORD-2027-610481,12/01/2027,2027-03-13,2027-08-03,Poland,-143.0
20,Negative date-diff,ORD-2027-599775,02/02/2027,2027-03-17,2027-08-03,Belgium,-139.0
36,Negative date-diff,ORD-2027-881543,2027-01-09,2027-03-24,2027-06-03,Germany,-71.0
0,Negative date-diff,ORD-2026-339139,2026-12-07,2027-02-24,2027-05-02,France,-67.0


## Data Quality Findings – Delivery vs Invoice Dates

-  The variety in date formats in Deliveries, appears to origin from the different delivery terminals used by courier companies. Since the logistic system, on the receiving end of the delivery flow, is old, it does not have the possibility to align date formats. This results in the inconsistent date representations.
-  Historical data shows that 44% of the orders comply  with the 10-day rule (0<= date_diff <=10). This indicates a significant gap to full compliance.
-  A notable share of orders (24%) show a negative date difference, where the invoice date precedes the delivery confirmation date. This may indicate differences in business processes or any inconsistency in time stamp across systems. 


















An analysis of the time difference between delivery_confirmed_date and invoice_date identified a number of cases with negative values, indicating that the invoice date precedes the delivery confirmation date.

**A subset of these cases was investigated manually in the source CSV files:**.<br>

*- Several records were confirmed to be data quality issues, primarily caused by:*.<br>
- Inconsistent or incorrect date formats.<br>
- Incorrectly recorded delivery dates.<br>
  
*- Other records could not be conclusively explained based on available data. Possible explanations include:* <br>

- Delayed registration of delivery confirmations in the logistics system (TMS).<br>
- Differences in business process (e.g. invoicing prior to delivery).<br>
- Integration or timing discrepancies between systems.<br>

*Further clarification would require:*
- Validation of business rules for invoicing vs delivery.<br>
- Review of source system definitions and event timestamps.<br>